In [2]:
from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.ollama import OllamaProvider
from watchfiles import awatch

model = OpenAIChatModel(
    model_name="llama3:latest",
    provider=OllamaProvider(base_url="http://localhost:11434/v1"),
)

agent = Agent(model=model, system_prompt="You are a helpful assistant")

result = await agent.run("What is th capital of Marche region in italy?")
print(result.output)

The Marche region in Italy has several provincial capitals, but the administrative seat and largest city in the region is Ancona. Ancona has been the capital of the Marche since 1860.


In [ ]:
# using Pydantic Ai UI to interact with the model
import uvicorn
from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.ollama import OllamaProvider

model = OpenAIChatModel(
    model_name="llama3:latest",
    provider=OllamaProvider(base_url="http://localhost:11434/v1"),
)

agent = Agent(model=model, system_prompt="You are a helpful assistant")
app = agent.to_web()

config = uvicorn.Config(app, host="127.0.0.1", port=8000, log_level="info")
server = uvicorn.Server(config)
await server.serve()

INFO:     Started server process [27152]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)


INFO:     127.0.0.1:59471 - "GET / HTTP/1.1" 200 OK
INFO:     127.0.0.1:59471 - "GET /api/configure HTTP/1.1" 200 OK
INFO:     127.0.0.1:59869 - "GET /api/configure HTTP/1.1" 200 OK
INFO:     127.0.0.1:59870 - "POST /api/chat HTTP/1.1" 200 OK
INFO:     127.0.0.1:60007 - "GET /api/configure HTTP/1.1" 200 OK
INFO:     127.0.0.1:60039 - "GET /api/configure HTTP/1.1" 200 OK
INFO:     127.0.0.1:60039 - "GET /api/configure HTTP/1.1" 200 OK
INFO:     127.0.0.1:60054 - "GET /api/configure HTTP/1.1" 200 OK


In [14]:
# structured output using mistral
from pydantic import BaseModel
from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.ollama import OllamaProvider

class CityModel(BaseModel):
    city: str
    country: str
    population: int
    fun_fact: str


model = OpenAIChatModel(
    model_name="mistral:latest",
    provider=OllamaProvider(base_url="http://localhost:11434/v1"),
)

agent = Agent(
    model=model,
    output_type=CityModel,
    system_prompt="You are a geography expert."
)

result = await agent.run("Tell me about rome in italy ")
print(result.output.city)
print(result.output.country)
print(result.output.population)
print(result.output.fun_fact)


Rome
Italy
2853000
Rome is the capital city of Italy and one of the oldest continually inhabited cities in Europe.


In [15]:
# tools
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.ollama import OllamaProvider
from pydantic_ai import Agent
from datetime import datetime


model = OpenAIChatModel(
    model_name="qwen2.5:latest",
    provider=OllamaProvider(base_url="http://localhost:11434/v1"),
)

agent = Agent(
    model=model,
    system_prompt="You are a geography expert."
)

@agent.tool_plain
def get_current_time() -> str:
    """Returns the current time."""
    return datetime.now().strftime("%H:%M:%S")


@agent.tool_plain
def get_weather(city: str) -> str:
    """Gets the weather for a city."""
    return f"It's sunny and 22°C in {city}"

result = await agent.run("What time is it and what's the weather in Rome?")
print(result.output)

The current time is 09:20:57. The weather in Rome is sunny with a temperature of 22°C.


In [9]:
# message history
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.ollama import OllamaProvider
from pydantic_ai import Agent

model = OpenAIChatModel(
    model_name="qwen2.5:latest",
    provider=OllamaProvider(base_url="http://localhost:11434/v1"),
)

agent = Agent(model=model)

result1 = await agent.run("My name is Antonio")
print(f"Turn 1: {result1.output}")

history = result1.all_messages()

result2 = await agent.run("What is my name?", message_history=history)
print(f"Turn 2: {result2.output}")

Turn 1: Hello Antonio! Nice to meet you. How can I assist you today? Feel free to ask me any questions or let me know if you have any topics you'd like to discuss.
Turn 2: Your name is Antonio.


In [17]:
# dependency injection

from pydantic_ai import RunContext
from dataclasses import dataclass
from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.ollama import OllamaProvider

@dataclass
class UserContext:
    username: str
    is_premium: bool

model = OpenAIChatModel(
    model_name="qwen2.5:latest",
    provider=OllamaProvider(base_url="http://localhost:11434/v1"),
)

agent = Agent(
    model=model,
    deps_type=UserContext,
    system_prompt="You are a helpful assistant."
)


@agent.system_prompt
def personalize(ctx: RunContext[UserContext]) -> str:
    tier = "premium" if ctx.deps.is_premium else "free"
    return f"The user is {ctx.deps.username} on the {tier} plan."

user = UserContext(username="Antonio", is_premium=True)
result = await agent.run("What can i do?", deps=user)
print(result.output)


It sounds like you're asking about what options or opportunities are available to you as an Antoni0 on the premium plan. Given that you have the premium plan, here are some general benefits and actions you might be able to take:

1. **Access Exclusive Content**: Premium plans often come with additional content or features not available in free tiers.

2. **Enhanced Features**: Utilize any advanced tools, support for multiple devices, or personalized services offered by the service provider.

3. **Customer Support**: Enjoy priority or 24/7 customer support if your plan includes this benefit.

4. **Subscriptions and Services**: If applicable, subscribe to additional services that complement your current plan.

5. **Customization Options**: Take advantage of any customization options available for settings, notifications, etc.

6. **Events and Webinars**: Participate in any exclusive events or webinars hosted by the service provider.

7. **Discounts and Offers**: Be aware of any discounts

In [18]:
# streaming response
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.ollama import OllamaProvider
from pydantic_ai import Agent

model = OpenAIChatModel(
    model_name="qwen2.5:latest",
    provider=OllamaProvider(base_url="http://localhost:11434/v1"),
)

agent = Agent(model=model)

async with agent.run_stream("Write a short poem about Rome") as response:
    async for chunk in response.stream_text():
        print(chunk, end="", flush=True)


In marble hallsIn marble halls and cobbleIn marble halls and cobblestone,
WhereIn marble halls and cobblestone,
Where empires ofIn marble halls and cobblestone,
Where empires of old were knownIn marble halls and cobblestone,
Where empires of old were known,
RomeIn marble halls and cobblestone,
Where empires of old were known,
Rome stands, aIn marble halls and cobblestone,
Where empires of old were known,
Rome stands, a grand monument,
In marble halls and cobblestone,
Where empires of old were known,
Rome stands, a grand monument,
To power,In marble halls and cobblestone,
Where empires of old were known,
Rome stands, a grand monument,
To power, art, andIn marble halls and cobblestone,
Where empires of old were known,
Rome stands, a grand monument,
To power, art, and human boon.

In marble halls and cobblestone,
Where empires of old were known,
Rome stands, a grand monument,
To power, art, and human boon.

Through ages,In marble halls and cobblestone,
Where empires of old were known,
Rom